# Análisis completo de identificadores — ECV 2025

**Objetivo:** establecer, con evidencia para cada uno de los 17 archivos de la ECV 2025, qué columnas identifican una vivienda, un hogar y una persona, antes de consolidar la información a nivel hogar.

**Alcance:** solo se analizan las columnas llave (`DIRECTORIO`, `SECUENCIA_ENCUESTA`, `SECUENCIA_P`, `ORDEN`) y algunas auxiliares (`P3204`, `P8692S3A1`, `CANT_PERSONAS_HOGAR`, `FEX_C`). **No se consolida ninguna variable**; los únicos cruces comparan llaves entre archivos.

Este análisis reemplaza la conclusión sobre llaves de `consolidate_dfs.ipynb`; las correcciones puntuales están al final.

**Leyenda** usada en tablas y comentarios: `D` = `DIRECTORIO`, `SE` = `SECUENCIA_ENCUESTA`, `SP` = `SECUENCIA_P`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))

from config import config

In [2]:
CSV_DIR = config.EXTRACTED / "ECV_2025_extracted" / "csv"

KEY_COLS = ["DIRECTORIO", "SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN"]
SHORT = {
    "DIRECTORIO": "D",
    "SECUENCIA_ENCUESTA": "SE",
    "SECUENCIA_P": "SP",
    "ORDEN": "ORDEN",
}

# combinaciones que se usan a lo largo del notebook
D_SE = ["DIRECTORIO", "SECUENCIA_ENCUESTA"]
D_SP = ["DIRECTORIO", "SECUENCIA_P"]
D_SP_ORDEN = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]

# alias corto -> nombre del archivo, para no arrastrar tildes en el código
FILES = {
    "vivienda": "Datos de la vivienda.csv",
    "servicios": "Servicios del hogar.csv",
    "tenencia": (
        "Tenencia y financiacion de la vivienda que ocupa el hogar.csv"
    ),
    "condiciones": "Condiciones de vida del hogar y tenencia de bienes.csv",
    "adquisicion": "Adquisición de bienes y servicios.csv",
    "caracteristicas": "Características y composición del hogar.csv",
    "salud": "Salud.csv",
    "diseno": "Variables diseño muestral.csv",
    "educacion": "Educación.csv",
    "tic": "Tecnologías de información y comunicación.csv",
    "fuerza_trabajo": "Fuerza de trabajo.csv",
    "trabajo_infantil": "Trabajo infantil.csv",
    "menores_5": (
        "Atención integral de los niños y niñas menores de 5 años.csv"
    ),
    "item": "Adquisición de bienes y servicios por ítem.CSV",
    "programas": (
        "Condiciones de vida del hogar y tenencia de bienes (programas).csv"
    ),
    "escritura": (
        "Tenencia y financiacion de la vivienda que ocupa el hogar "
        "(personas escritura).csv"
    ),
    "pdet": "FEX_PDET_VIV_2025.csv",
}

# columnas auxiliares que se cargan junto con las llaves
EXTRA_COLS = {
    "servicios": ["CANT_PERSONAS_HOGAR"],
    "item": ["P3204"],
    "escritura": ["P8692S3A1"],
}

In [3]:
def read_keys(alias, extra=()):
    """Lee las columnas llave presentes en el archivo, más `extra`."""
    path = CSV_DIR / FILES[alias]
    header = pd.read_csv(path, sep=";", nrows=0).columns
    cols = [c for c in [*KEY_COLS, *extra] if c in header]
    # el archivo por ítem es el único que usa punto decimal
    decimal = "." if alias == "item" else ","
    return pd.read_csv(path, sep=";", usecols=cols, decimal=decimal)


def n_dups(df, key):
    """Filas repetidas según `key` (NaN si falta alguna columna)."""
    if not set(key) <= set(df.columns):
        return np.nan
    return int(df.duplicated(subset=key).sum())


def broken_sequences(df, group, col):
    """Número de grupos en los que `col` no es exactamente 1, 2, ..., n."""
    if not {*group, col} <= set(df.columns):
        return np.nan
    g = df.groupby(group)[col].agg(["min", "max", "nunique", "size"])
    ok = (
        (g["min"] == 1) & (g["max"] == g["size"]) & (g["nunique"] == g["size"])
    )
    return int((~ok).sum())


def key_set(df, cols):
    """Conjunto de tuplas de llave, para comparar archivos."""
    return set(df[cols].itertuples(index=False, name=None))


def orphans(child, child_key, parent, parent_key):
    """Combinaciones de llave del hijo que no existen en el padre."""
    left = child[child_key].drop_duplicates()
    right = parent[parent_key].drop_duplicates().set_axis(child_key, axis=1)
    check = left.merge(right, on=child_key, how="left", indicator=True)
    return int((check["_merge"] == "left_only").sum())

In [4]:
missing = [
    alias for alias, name in FILES.items() if not (CSV_DIR / name).exists()
]
print(len(FILES), missing)  # esperado 17 []

17 []


## Inventario de archivos

Se cargan únicamente las columnas llave (y las auxiliares de `EXTRA_COLS`) de los 17 archivos. Leer solo estas columnas mantiene la memoria baja incluso con el archivo por ítem, de 1,47 millones de filas.

In [5]:
keys = {alias: read_keys(alias, EXTRA_COLS.get(alias, ())) for alias in FILES}

In [6]:
rows = []
for alias, df in keys.items():
    present = [c for c in KEY_COLS if c in df]
    max_sp = df["SECUENCIA_P"].max() if "SECUENCIA_P" in df else None
    max_orden = df["ORDEN"].max() if "ORDEN" in df else None
    rows.append(
        {
            "archivo": alias,
            "filas": len(df),
            "llaves": " + ".join(SHORT[c] for c in present),
            "nulos_en_llaves": int(df[present].isna().sum().sum()),
            "max_SP": max_sp,
            "max_ORDEN": max_orden,
        }
    )

inventory = pd.DataFrame(rows).set_index("archivo").convert_dtypes()
display(inventory)

,filas,llaves,nulos_en_llaves,max_SP,max_ORDEN
archivo,,,,,
vivienda,86848,D + SE + SP + ORDEN,0,1,1
servicios,87060,D + SE + SP + ORDEN,0,1,6
tenencia,87060,D + SE + SP + ORDEN,0,1,6
condiciones,87060,D + SE + SP + ORDEN,0,1,6
adquisicion,87060,D + SE + SP + ORDEN,0,1,6
caracteristicas,235350,D + SE + SP + ORDEN,0,6,21
salud,235350,D + SE + SP + ORDEN,0,6,21
diseno,235350,D + SP + ORDEN,0,6,21
educacion,221043,D + SE + SP + ORDEN,0,6,20


- **Ningún archivo tiene nulos en las llaves.**
- `diseno` no trae `SECUENCIA_ENCUESTA` y `pdet` solo trae `DIRECTORIO`: una llave que dependa de `SE` no sirve para todos los archivos.
- Los tamaños ya sugieren niveles: 86.848 filas (`vivienda`), 87.060 (`servicios`, `tenencia`, `condiciones`, `adquisicion`), 235.350 (`caracteristicas`, `salud`, `diseno`) y subconjuntos más pequeños (módulos filtrados por edad y tablas hijas).
- `SECUENCIA_P` vale siempre 1 en `vivienda` y en los 4 archivos de 87.060 filas, y llega hasta 6 en los archivos de persona, el mismo máximo de hogares por vivienda que aparece más abajo.

## Buscar qué combinación identifica un registro en cada archivo

Para cada archivo se cuentan las filas repetidas bajo cada combinación candidata. Una llave válida debe dar **0**; `<NA>` indica que al archivo le falta alguna de las columnas.

In [7]:
CANDIDATES = {
    "D": ["DIRECTORIO"],
    "D+SE": D_SE,
    "D+SP": D_SP,
    "D+ORDEN": ["DIRECTORIO", "ORDEN"],
    "D+SE+SP": ["DIRECTORIO", "SECUENCIA_ENCUESTA", "SECUENCIA_P"],
    "D+SP+ORDEN": D_SP_ORDEN,
}

dup_matrix = pd.DataFrame(
    {
        name: {alias: n_dups(df, key) for alias, df in keys.items()}
        for name, key in CANDIDATES.items()
    }
).astype("Int64")
display(dup_matrix)

,D,D+SE,D+SP,D+ORDEN,D+SE+SP,D+SP+ORDEN
vivienda,0,0,0,0,0,0
servicios,212,0,212,0,0,0
tenencia,212,0,212,0,0,0
condiciones,212,0,212,0,0,0
adquisicion,212,0,212,0,0,0
caracteristicas,148502,454,148290,454,0,0
salud,148502,454,148290,454,0,0
diseno,148502,<NA>,148290,454,<NA>,0
educacion,134195,416,133983,416,0,0
tic,134195,416,133983,416,0,0


- **`vivienda` y `pdet`:** `D` ya es único, una fila por vivienda.
- **Archivos de hogar** (`servicios`, `tenencia`, `condiciones`, `adquisicion`): `D` repite 212 veces por las viviendas con varios hogares, y `D+SP` repite exactamente lo mismo porque `SP` es constante. **`D+SE` es único.**
- **Archivos de persona:** `D+SE` y `D+ORDEN` repiten (454 veces en los archivos completos) y **`D+SP+ORDEN` es único en todos**, incluido `diseno`. `D+SE+SP` también da 0, pero solo porque ahí `SE` y `ORDEN` contienen lo mismo (siguiente sección), y no se puede armar en `diseno`.
- **Tablas hijas** (`item`, `programas`, `escritura`): `D+SP+ORDEN` es único, pero ojo, **`D+SE` también da 0 repetidos**. Que una combinación sea única no la convierte en llave de hogar; la siguiente sección muestra qué numera `SE` en estas tablas.
- `D+SP+ORDEN` da 0 en **todos** los archivos que tienen esas columnas, también en los de hogar. Es la pista de una regla general: cada registro es la posición `ORDEN` dentro del padre `D+SP`.

## Examinar la relación entre SECUENCIA_ENCUESTA, SECUENCIA_P y ORDEN

Para entender qué numera cada columna se mide:

- `filas_SE!=ORDEN`: filas donde `SECUENCIA_ENCUESTA` y `ORDEN` difieren.
- `valores_SP`: cuántos valores distintos toma `SECUENCIA_P`.
- `*_rota_en_*`: en cuántos grupos la numeración **no** es exactamente 1, 2, ..., n. Un 0 significa que la columna numera los registros dentro de ese grupo.

In [8]:
se = "SECUENCIA_ENCUESTA"

rows = []
for alias, df in keys.items():
    if "ORDEN" not in df:
        continue  # pdet solo tiene DIRECTORIO
    rows.append(
        {
            "archivo": alias,
            "filas_SE!=ORDEN": (
                int((df[se] != df["ORDEN"]).sum()) if se in df else None
            ),
            "valores_SP": df["SECUENCIA_P"].nunique(),
            "ORDEN_rota_en_D+SP": broken_sequences(df, D_SP, "ORDEN"),
            "SE_rota_en_D+SP": broken_sequences(df, D_SP, se),
            "SE_rota_en_D": broken_sequences(df, ["DIRECTORIO"], se),
        }
    )

relations = pd.DataFrame(rows).set_index("archivo").convert_dtypes()
display(relations)

,filas_SE!=ORDEN,valores_SP,ORDEN_rota_en_D+SP,SE_rota_en_D+SP,SE_rota_en_D
archivo,,,,,
vivienda,0,1,0,0,0
servicios,0,1,0,0,0
tenencia,0,1,0,0,0
condiciones,0,1,0,0,0
adquisicion,0,1,0,0,0
caracteristicas,0,6,0,0,181
salud,0,6,0,0,181
diseno,<NA>,6,0,<NA>,<NA>
educacion,0,6,1246,1246,1421


- **Archivos de hogar:** `SP` toma un solo valor (1) y `SE = ORDEN` numera los hogares 1..n dentro de cada vivienda, sin saltos. **El padre del hogar es la vivienda.**
- **`caracteristicas`, `salud`, `diseno`:** `SE = ORDEN` y la numeración reinicia en cada `D+SP`: **`SE`/`ORDEN` numeran personas y `SP` es el hogar.** Dentro de la vivienda la numeración se rompe exactamente en las 181 viviendas con más de un hogar, SE se rompe pues se toma solo la vivienda y se omite el hogar, luego se identifican 181 registros rotos que corresponden a numeraciones al nivel de persona que no siguen la forma 1,2,.. .
- **Módulos filtrados por edad** (`educacion`, `tic`, `fuerza_trabajo`, `trabajo_infantil`, `menores_5`): `ORDEN` tiene huecos dentro del hogar porque faltan las personas a las que no aplica el módulo; en `trabajo_infantil` y `menores_5` la numeración no es 1..n en ningún hogar. **`ORDEN` conserva el número de la persona en el hogar, no se renumera por módulo** (tabla) , y por eso sirve para enlazar módulos.
- **`item`:** `ORDEN` reinicia en cada hogar, pero `SE` sigue contando a lo largo de la vivienda: 3.400 filas con `SE != ORDEN` y 211 hogares donde `SE` no es 1..n. En `programas` y `escritura`, `SE = ORDEN` numera filas (programas o titulares), no hogares. Por eso `D+SE` resultaba único en estas tablas.
- **`SECUENCIA_ENCUESTA` no significa lo mismo en todos los archivos.** La posición confiable es `ORDEN` dentro de `D+SP`.


Al parecer SE en algunas tablas contiene la misma info que ORDEN

## Validar la jerarquía vivienda → hogar → persona

Si la hipótesis es correcta, cada nivel debe encajar exactamente en el anterior.

### Hogar → vivienda

In [9]:
HOUSEHOLD_FILES = ["servicios", "tenencia", "condiciones", "adquisicion"]

dwellings = set(keys["vivienda"]["DIRECTORIO"])
print(len(dwellings))  # esperado 86848

for alias in HOUSEHOLD_FILES:
    same = set(keys[alias]["DIRECTORIO"]) == dwellings
    print(f"{alias:12s} mismas viviendas que 'vivienda': {same}")

86848
servicios    mismas viviendas que 'vivienda': True
tenencia     mismas viviendas que 'vivienda': True
condiciones  mismas viviendas que 'vivienda': True
adquisicion  mismas viviendas que 'vivienda': True


In [10]:
households_per_dwelling = keys["servicios"].groupby("DIRECTORIO").size()
display(
    households_per_dwelling.value_counts()
    .sort_index()
    .rename_axis("hogares_en_la_vivienda")
    .to_frame("viviendas")
)

shared = int((households_per_dwelling > 1).sum())
extra = int((households_per_dwelling - 1).sum())
print(f"viviendas con más de un hogar: {shared}")  # esperado 181
print(f"hogares adicionales: {extra}")  # esperado 212

,viviendas
hogares_en_la_vivienda,
1,86667
2,157
3,20
4,2
5,1
6,1


viviendas con más de un hogar: 181
hogares adicionales: 212


`shared = int((households_per_dwelling > 1).sum())`: Suma las viviendas que tienen estrictamente más de 1 hogar.

`extra = int((households_per_dwelling - 1).sum())`: Para cada vivienda calcula los hogares "adicionales" ($N - 1$) y los suma todos.(Ejemplo: Una vivienda con 3 hogares aporta $3 - 1 = 2$ hogares adicionales).

Los 4 archivos de hogar cubren exactamente las 86.848 viviendas.

**Corrección a `consolidate_dfs.ipynb`:** allí se leyó el 212 como *212 viviendas compartidas*. En realidad hay **181 viviendas con más de un hogar**; 212 es el número de **hogares adicionales**: las 157 viviendas con 2 hogares aportan 157, las 20 con 3 aportan 40, las 2 con 4 aportan 6, y las viviendas con 5 y 6 hogares aportan 4 y 5.

* **181 viviendas con más de un hogar**
* **212 hogares adicionales** (que van extra a los hogares  que se tendiran si una vivienda es solo un hogar)

### Los mismos hogares en todos los archivos de hogar

In [11]:
households = key_set(keys["servicios"], D_SE)
print(len(households))  # esperado 87060

for alias in HOUSEHOLD_FILES:
    same = key_set(keys[alias], D_SE) == households
    print(f"{alias:12s} mismos hogares que 'servicios': {same}")

87060
servicios    mismos hogares que 'servicios': True
tenencia     mismos hogares que 'servicios': True
condiciones  mismos hogares que 'servicios': True
adquisicion  mismos hogares que 'servicios': True


### Hogar en los archivos hijos

Si `SP` apunta al hogar, todo par `(D, SP)` de un archivo de persona o de una tabla hija debe existir como `(D, SE)` en `servicios`: **0 huérfanos**. Como contraste se prueba `(D, SE)` en el archivo hijo, la llave de hogar que se venía usando en `consolidate_dfs.ipynb`.

In [12]:
PERSON_FILES = [
    "caracteristicas",
    "salud",
    "diseno",
    "educacion",
    "tic",
    "fuerza_trabajo",
    "trabajo_infantil",
    "menores_5",
]
CHILD_FILES = [*PERSON_FILES, "item", "programas", "escritura"]
parent = keys["servicios"]

rows = []
for alias in CHILD_FILES:
    df = keys[alias]
    has_se = "SECUENCIA_ENCUESTA" in df
    rows.append(
        {
            "archivo": alias,
            "hogares_cubiertos": len(key_set(df, D_SP)),
            "huerfanos_con_D+SP": orphans(df, D_SP, parent, D_SE),
            "huerfanos_con_D+SE": (
                orphans(df, D_SE, parent, D_SE) if has_se else None
            ),
        }
    )

display(pd.DataFrame(rows).set_index("archivo").convert_dtypes())

,hogares_cubiertos,huerfanos_con_D+SP,huerfanos_con_D+SE
archivo,,,
caracteristicas,87060,0,147864
salud,87060,0,147864
diseno,87060,0,<NA>
educacion,87060,0,133599
tic,87060,0,133599
fuerza_trabajo,87060,0,107698
trabajo_infantil,20703,0,25921
menores_5,12736,0,14291
item,86675,0,1386532


- **Los 11 archivos hijos tienen 0 huérfanos con `(D, SP)`:** todo hogar al que apunta `SP` existe en `servicios`.
- Con `(D, SE)` aparecen miles de huérfanos (147.864 en `caracteristicas`). En esos archivos `SE` es el número de la persona: en una vivienda con un solo hogar, la persona 3 se leería como el "hogar 3", que no existe. Hasta en `programas` sobran 7 pares.
- `educacion`, `tic` y `fuerza_trabajo` omiten personas pero siguen cubriendo los 87.060 hogares; `trabajo_infantil` cubre 20.703, `menores_5` 12.736, `escritura` 3.403 y `programas` 842. `item` cubre 86.675: **385 hogares no reportan ningún ítem**.

### Prueba decisiva: tamaño del hogar

`servicios` trae `CANT_PERSONAS_HOGAR`. Si `(D, SP)` es el hogar en los archivos de persona completos, contar personas por `(D, SP)` debe reproducir ese número en el 100 % de los hogares.

In [13]:
size = keys["servicios"].set_index(D_SE)["CANT_PERSONAS_HOGAR"]

for alias in ["caracteristicas", "salud", "diseno"]:
    df = keys[alias]
    by_sp = df.groupby(D_SP).size().rename_axis(D_SE).reindex(size.index)
    share_sp = (by_sp == size).mean()
    if "SECUENCIA_ENCUESTA" in df:
        by_se = df.groupby(D_SE).size().reindex(size.index)
        share_se = f"{(by_se == size).mean():.4f}"
    else:
        share_se = "no aplica"
    print(f"{alias:16s} (D, SP): {share_sp:.4f}   (D, SE): {share_se}")
# debe dar 1.0000 con (D, SP)

print(f"hogares de una sola persona: {(size == 1).mean():.4f}")

caracteristicas  (D, SP): 1.0000   (D, SE): 0.2198
salud            (D, SP): 1.0000   (D, SE): 0.2198
diseno           (D, SP): 1.0000   (D, SE): no aplica
hogares de una sola persona: 0.2193


**Contar personas por `(D, SP)` reproduce `CANT_PERSONAS_HOGAR` en el 100 % de los hogares** de los tres archivos de persona completos. Con `(D, SE)` solo coincide el 22 %, prácticamente la proporción de hogares de una sola persona (21,9 %): en un hogar de una persona la cuenta equivocada da 1 por casualidad.

### La misma persona en todos los archivos de persona

In [14]:
people = key_set(keys["caracteristicas"], D_SP_ORDEN)

rows = []
for alias in PERSON_FILES:
    persons = key_set(keys[alias], D_SP_ORDEN)
    rows.append(
        {
            "archivo": alias,
            "personas": len(persons),
            "hogares": len(key_set(keys[alias], D_SP)),
            "contenidas_en_caracteristicas": persons <= people,
            "igual_a_caracteristicas": persons == people,
        }
    )

display(pd.DataFrame(rows).set_index("archivo"))

,personas,hogares,contenidas_en_caracteristicas,igual_a_caracteristicas
archivo,,,,
caracteristicas,235350,87060,True,True
salud,235350,87060,True,True
diseno,235350,87060,True,True
educacion,221043,87060,True,False
tic,221043,87060,True,False
fuerza_trabajo,195074,87060,True,False
trabajo_infantil,25969,20703,True,False
menores_5,14307,12736,True,False


- **Toda persona de cualquier módulo existe en `caracteristicas`** con la misma llave `D+SP+ORDEN`.
- `salud` y `diseno` tienen exactamente las mismas 235.350 personas que `caracteristicas`.
- Los demás módulos son subconjuntos, así que al llevarlos al hogar habrá hogares o personas sin registro en ese módulo.

## Contraejemplo: por qué [DIRECTORIO, SECUENCIA_ENCUESTA] falla a nivel persona

In [15]:
design = keys["diseno"]
unique_d_orden = len(key_set(design, ["DIRECTORIO", "ORDEN"]))
salud_dups = n_dups(keys["salud"], D_SE)

print(f"filas en diseno: {len(design)}")  # esperado 235350
print(f"pares únicos (D, ORDEN): {unique_d_orden}")  # esperado 234896
print(f"diferencia: {len(design) - unique_d_orden}")  # esperado 454
print(f"repetidos de (D, SE) en salud: {salud_dups}")  # esperado 454

filas en diseno: 235350
pares únicos (D, ORDEN): 234896
diferencia: 454
repetidos de (D, SE) en salud: 454


El `234896` de `consolidate_dfs.ipynb` es el número de **pares únicos `(D, ORDEN)`**. Los 454 que faltan para llegar a 235.350 coinciden con los 454 repetidos de `(D, SE)` en `salud`: son personas de viviendas con más de un hogar, donde cada hogar numera a sus miembros desde 1 y el número de persona se repite dentro de la vivienda.

Ejemplo con una vivienda de dos hogares:

In [16]:
example = households_per_dwelling[households_per_dwelling == 2].index[0]
print(f"DIRECTORIO de ejemplo: {example}")

services = keys["servicios"]
display(services[services["DIRECTORIO"] == example])

DIRECTORIO de ejemplo: 8370223


,DIRECTORIO,SECUENCIA_ENCUESTA,SECUENCIA_P,ORDEN,CANT_PERSONAS_HOGAR
562,8370223,1,1,1,3
563,8370223,2,1,2,2


In [17]:
characteristics = keys["caracteristicas"]
display(characteristics[characteristics["DIRECTORIO"] == example])

,DIRECTORIO,SECUENCIA_ENCUESTA,SECUENCIA_P,ORDEN
1522,8370223,1,1,1
1523,8370223,2,1,2
1524,8370223,3,1,3
1525,8370223,1,2,1
1526,8370223,2,2,2


- En `servicios` la vivienda 8370223 tiene dos hogares, `SE = ORDEN` = 1 (3 personas) y 2 (2 personas), ambos con `SP = 1`.
- En `caracteristicas` las personas del primer hogar tienen `SP = 1` y `ORDEN` 1, 2, 3; las del segundo tienen `SP = 2` y `ORDEN` 1, 2.
- Con `(D, SE)` los pares `(8370223, 1)` y `(8370223, 2)` se repiten, y además se confunden con los hogares 1 y 2 de `servicios`. **Con `(D, SP, ORDEN)` cada persona es única y su `SP` coincide con el `SE` de su hogar.**

## Examinar archivos especiales

Tres tablas no son ni de hogar ni de persona, y `pdet` solo tiene `DIRECTORIO`.

### Adquisición de bienes y servicios por ítem

In [18]:
item = keys["item"]
print(len(item))  # esperado 1473207

for key in [[*D_SP, "P3204"], D_SP_ORDEN, D_SE]:
    print(key, n_dups(item, key))

1473207
['DIRECTORIO', 'SECUENCIA_P', 'P3204'] 0
['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'] 0
['DIRECTORIO', 'SECUENCIA_ENCUESTA'] 0


In [19]:
orden_broken = broken_sequences(item, D_SP, "ORDEN")
se_broken = broken_sequences(item, ["DIRECTORIO"], "SECUENCIA_ENCUESTA")

print(f"hogares donde ORDEN no es 1..n: {orden_broken}")  # esperado 0
print(f"viviendas donde SE no es 1..n: {se_broken}")
print(f"hogares con ítems: {len(key_set(item, D_SP))} de {len(households)}")

hogares donde ORDEN no es 1..n: 0
viviendas donde SE no es 1..n: 1
hogares con ítems: 86675 de 87060


In [20]:
# la misma vivienda de dos hogares del contraejemplo
item_cols = ["SECUENCIA_ENCUESTA", "ORDEN"]
display(
    item[item["DIRECTORIO"] == example]
    .groupby("SECUENCIA_P")[item_cols]
    .agg(["min", "max"])
)

SECUENCIA_ENCUESTA     ORDEN    
                           min max   min max
SECUENCIA_P                                 
1                            1  14     1  14
2                           15  30     1  16

- Un ítem se identifica por **`D+SP+P3204`** (hogar + código del ítem). `D+SP+ORDEN` también es único porque `ORDEN` numera los ítems 1..n dentro de cada hogar.
- `D+SE` es único, pero `SE` es un contador de filas **de toda la vivienda**: en la vivienda 8370223 los ítems del segundo hogar siguen la numeración del primero (`SE` 15 a 30) mientras `ORDEN` vuelve a empezar en 1. De ahí salen los 3.400 casos con `SE != ORDEN`; solo 1 vivienda rompe ese contador.
- **385 hogares no aparecen en este archivo.**

### Condiciones de vida del hogar (programas)

In [21]:
programs = keys["programas"]
print(f"filas: {len(programs)}, hogares: {len(key_set(programs, D_SP))}")
print(f"repetidos con D+SP+ORDEN: {n_dups(programs, D_SP_ORDEN)}")  # 0

display(
    programs.groupby(D_SP)
    .size()
    .value_counts()
    .rename_axis("programas_en_el_hogar")
    .to_frame("hogares")
)

bad_se = orphans(programs, D_SE, keys["servicios"], D_SE)
print(f"pares (D, SE) que no son un hogar: {bad_se}")

filas: 849, hogares: 842
repetidos con D+SP+ORDEN: 0


,hogares
programas_en_el_hogar,
1,835
2,7


pares (D, SE) que no son un hogar: 7


- 849 programas en 842 hogares: 7 hogares reportan 2 programas. La llave es **`D+SP+ORDEN`**, con `ORDEN` = número del programa dentro del hogar.
- Los 7 pares `(D, SE)` que no corresponden a ningún hogar vuelven a mostrar que en esta tabla `SE` numera filas, no hogares.

### Tenencia y financiación de la vivienda (personas escritura)

In [22]:
deed = keys["escritura"]
print(f"filas: {len(deed)}, hogares: {len(key_set(deed, D_SP))}")
print(f"repetidos con D+SP+ORDEN: {n_dups(deed, D_SP_ORDEN)}")  # esperado 0

orden_broken = broken_sequences(deed, D_SP, "ORDEN")
print(f"hogares donde ORDEN no es 1..n: {orden_broken}")  # esperado 0

filas: 7030, hogares: 3403
repetidos con D+SP+ORDEN: 0
hogares donde ORDEN no es 1..n: 0


In [23]:
differ = deed[deed["ORDEN"] != deed["P8692S3A1"]]
print(f"filas con ORDEN != P8692S3A1: {len(differ)}")  # esperado 270
display(differ.head(6))

filas con ORDEN != P8692S3A1: 270


,DIRECTORIO,SECUENCIA_ENCUESTA,SECUENCIA_P,ORDEN,P8692S3A1
14,8369779,1,1,1,2
15,8369779,2,1,2,4
17,8369835,2,1,2,3
65,8370458,1,1,1,2
66,8370458,2,1,2,1
73,8370501,1,1,1,2


In [24]:
D_SP_PERSON = ["DIRECTORIO", "SECUENCIA_P", "P8692S3A1"]
print(f"repetidos con D+SP+P8692S3A1: {n_dups(deed, D_SP_PERSON)}")  # 0

referenced = key_set(deed, D_SP_PERSON)
print(f"personas referenciadas que existen: {referenced <= people}")  # True

repetidos con D+SP+P8692S3A1: 0
personas referenciadas que existen: True


- 7.030 titulares en 3.403 hogares, llave **`D+SP+ORDEN`**, con `ORDEN` = número de fila 1..n dentro del hogar.
- `P8692S3A1` guarda un número de persona: en 270 filas no coincide con `ORDEN` (por ejemplo, la fila 1 apunta a la persona 2 y la fila 2 a la persona 4).
- `D+SP+P8692S3A1` es única y todas esas personas existen en `caracteristicas`. **Para enlazar un titular con su registro de persona se usa `P8692S3A1`, no `ORDEN`.** Es una hipótesis fuerte, pero hay que confirmar en el diccionario de datos que `P8692S3A1` es el número de orden de la persona.

### Factores de expansión PDET

In [25]:
pdet = keys["pdet"]
print(f"filas: {len(pdet)}")  # esperado 14900
print(f"DIRECTORIO único: {pdet['DIRECTORIO'].is_unique}")  # True
print(f"contenidas en 'vivienda': {set(pdet['DIRECTORIO']) <= dwellings}")

filas: 14900
DIRECTORIO único: True
contenidas en 'vivienda': True


**`pdet` es un subconjunto de 14.900 viviendas** (municipios PDET) con `DIRECTORIO` único; se enlaza directamente con `vivienda`.

## Examinar el factor de expansión FEX_C

`FEX_C` aparece en casi todos los archivos junto a las llaves, así que conviene verificar si sirve para cruzar.

In [26]:
raw = pd.read_csv(
    CSV_DIR / FILES["servicios"], sep=";", usecols=["FEX_C"], nrows=3
)
print(raw["FEX_C"].dtype, raw["FEX_C"].tolist())  # sin decimal="," es texto

str ['472,1319014', '308,8543189', '428,5162391']


In [27]:
fex = {
    alias: read_keys(alias, ["FEX_C"])
    for alias in [*HOUSEHOLD_FILES, "vivienda", "caracteristicas", "diseno"]
}

for alias in ["caracteristicas", "diseno"]:
    per_household = fex[alias].groupby(D_SP)["FEX_C"].nunique().max()
    print(f"{alias:16s} máx. valores de FEX_C por hogar: {per_household}")
# esperado 1: constante dentro del hogar

caracteristicas  máx. valores de FEX_C por hogar: 1
diseno           máx. valores de FEX_C por hogar: 1


In [28]:
base = fex["servicios"].set_index(D_SE)["FEX_C"]

rows = []
for alias in [*HOUSEHOLD_FILES[1:], "caracteristicas", "diseno"]:
    df = fex[alias]
    if alias in HOUSEHOLD_FILES:
        other = df.set_index(D_SE)["FEX_C"]
    else:
        other = df.groupby(D_SP)["FEX_C"].first().rename_axis(D_SE)
    diff = (other.reindex(base.index) - base).abs()
    rows.append(
        {
            "archivo": alias,
            "hogares_con_FEX_C_distinto": int((diff > 0).sum()),
            "max_diferencia": diff.max(),
        }
    )

# vivienda: se compara con el primer hogar de cada vivienda
dwelling_fex = fex["vivienda"].set_index("DIRECTORIO")["FEX_C"]
first_household = base.groupby(level="DIRECTORIO").first()
diff = (dwelling_fex.reindex(first_household.index) - first_household).abs()
rows.append(
    {
        "archivo": "vivienda",
        "hogares_con_FEX_C_distinto": int((diff > 0).sum()),
        "max_diferencia": diff.max(),
    }
)

display(pd.DataFrame(rows).set_index("archivo"))

,hogares_con_FEX_C_distinto,max_diferencia
archivo,,
tenencia,0,0.000000
condiciones,4553,0.000001
adquisicion,0,0.000000
caracteristicas,0,0.000000
diseno,4553,0.000001
vivienda,0,0.000000


In [29]:
pdet_fex = read_keys("pdet", ["FEX_C"]).set_index("DIRECTORIO")["FEX_C"]
weights = pd.DataFrame(
    {"pdet": pdet_fex, "nacional": dwelling_fex.reindex(pdet_fex.index)}
)
display(weights.describe())

,pdet,nacional
count,14900.000000,14900.000000
mean,123.232258,125.026489
std,152.420830,137.887967
min,4.401471,4.526068
25%,39.472772,34.417464
50%,75.273179,79.916514
75%,150.011731,161.239465
max,3424.524633,1546.215007


- Sin `decimal=","` el factor se lee como texto (`str` en pandas 3).
- Dentro de un hogar `FEX_C` es constante en los archivos de persona.
- `tenencia`, `adquisicion`, `caracteristicas` y `vivienda` repiten exactamente el `FEX_C` de `servicios`, pero **`condiciones` y `diseno` difieren en 4.553 hogares**, con diferencias de hasta 0,000001 (redondeo). Un cruce que incluya `FEX_C` perdería esos hogares sin avisar.
- El factor PDET no es una copia del nacional: tiene otra distribución (máximo 3.424 frente a 1.546), así que es un peso distinto, pensado para los municipios PDET.
- **`FEX_C` no es llave:** se toma de un solo archivo.

## Conclusión

La hipótesis se confirma en los 17 archivos. **Regla general: cada registro es la posición `ORDEN` dentro de su padre `(DIRECTORIO, SECUENCIA_P)`.** En los archivos de hogar el padre es la vivienda (`SP = 1`) y `ORDEN = SE` es el número del hogar; en los demás archivos `SP` es el número del hogar. `SECUENCIA_ENCUESTA` cambia de significado según el archivo, así que solo se usa en los archivos de hogar.

| Nivel | Archivos | Llave única | Llave hacia el padre |
| --- | --- | --- | --- |
| vivienda | `vivienda`, `pdet` | `DIRECTORIO` | — |
| hogar | `servicios`, `tenencia`, `condiciones`, `adquisicion` | `DIRECTORIO` + `SECUENCIA_ENCUESTA` | `DIRECTORIO` |
| persona | `caracteristicas`, `salud`, `diseno`, `educacion`, `tic`, `fuerza_trabajo`, `trabajo_infantil`, `menores_5` | `DIRECTORIO` + `SECUENCIA_P` + `ORDEN` | `DIRECTORIO` + `SECUENCIA_P` |
| ítem | `item` | `DIRECTORIO` + `SECUENCIA_P` + `P3204` | `DIRECTORIO` + `SECUENCIA_P` |
| programa | `programas` | `DIRECTORIO` + `SECUENCIA_P` + `ORDEN` | `DIRECTORIO` + `SECUENCIA_P` |
| titular de escritura | `escritura` | `DIRECTORIO` + `SECUENCIA_P` + `ORDEN` | hogar: `DIRECTORIO` + `SECUENCIA_P`; persona: `DIRECTORIO` + `SECUENCIA_P` + `P8692S3A1` |

**Reglas que salen del análisis**

- El hogar se llama `(DIRECTORIO, SECUENCIA_ENCUESTA)` en los archivos de hogar y `(DIRECTORIO, SECUENCIA_P)` en todos los demás. Como los nombres no coinciden, cualquier cruce necesita `left_on`/`right_on` explícitos: un `merge` sobre columnas homónimas mezcla hogares y personas sin dar error.
- `ORDEN` conserva el número de la persona en todos los módulos de persona.
- `SECUENCIA_ENCUESTA` no se usa fuera de los archivos de hogar.
- `FEX_C` no es llave; se toma de un solo archivo.
- Totales de control: 86.848 viviendas, 87.060 hogares, 235.350 personas.

**Correcciones a `consolidate_dfs.ipynb`**

- La llave de hogar `[DIRECTORIO, SECUENCIA_ENCUESTA]` solo vale en los archivos de hogar; en los de persona el hogar es `[DIRECTORIO, SECUENCIA_P]` (la celda que define `KH_salud` ya lo usaba).
- La llave de persona es `[DIRECTORIO, SECUENCIA_P, ORDEN]`, no `[DIRECTORIO, SECUENCIA_ENCUESTA, SECUENCIA_P]`: esta última solo funciona donde `SE = ORDEN` y no existe en `diseno`.
- La idea de *usar siempre `SECUENCIA_ENCUESTA`* rompe el cruce: en los archivos de persona `SE` es el número de la persona.
- No hay 212 viviendas compartidas sino **181**; 212 son hogares adicionales.
- `234896` es el número de pares únicos `(DIRECTORIO, ORDEN)`; la diferencia de 454 con 235.350 son personas de viviendas compartidas.

**Pendiente:** confirmar en el diccionario de datos que `P8692S3A1` es el número de orden de la persona que figura en la escritura.